# ClimaCity Paris -- Jour 2
## Spark SQL, Delta Lake et Structured Streaming

**Module** : Traitement de données massives avec Apache Spark et PySpark  
**Durée** : 1 journée (6 heures effectives)  
**Prérequis** : Avoir complété le Jour 1 -- la table `disponibilite_consolidee.parquet` doit être présente dans `data/output/`

---

Ce notebook couvre l'intégralité du Jour 2 du projet ClimaCity Paris.  
Il se divise en deux grandes parties :

- **Partie 1 -- Matin (3 h)** : Spark SQL et l'API de fenêtrage analytique, puis Delta Lake
  pour la persistance transactionnelle (écriture, time-travel, `MERGE INTO`).
- **Partie 2 -- Après-midi (3 h)** : Structured Streaming -- connexion à un flux simulé
  de mises à jour de stations, agrégations sur fenêtres glissantes, gestion des données
  tardives (late data) et déclenchement d'alertes.

> **Convention** : les cellules `# [EXERCICE]` contiennent une consigne à compléter.  
> Les cellules `# [CORRECTION]` proposent une solution -- ne les regardez qu'après avoir tenté.


---
## Section 0 -- Configuration

Même structure de chemins qu'au Jour 1. La table consolidée produite hier est le
point de départ de toutes les analyses.


In [35]:
from pathlib import Path
import time

# ── Chemins (même logique que Sessions 1 et 2) ───────────────────────────────
# "data" = racine du projet ; "../data" = fallback si cwd différent
_candidats = [Path("data"), Path("../data")]
DATA_DIR = next((p for p in _candidats if (p / "velib").exists()), _candidats[0])
OUTPUT_DIR = DATA_DIR / "output"
VELIB_CONSOLIDE = OUTPUT_DIR / "disponibilite_consolidee.parquet"
DELTA_DISPONIBLE = OUTPUT_DIR / "delta" / "disponibilite"
DELTA_ALERTES = OUTPUT_DIR / "delta" / "alertes"
STREAM_SOURCE_DIR = OUTPUT_DIR / "stream_input"    # répertoire surveillé par Spark
STREAM_CHECKPOINT = OUTPUT_DIR / "checkpoints"

for p in [VELIB_CONSOLIDE]:
    assert p.exists(), (
        f"Fichier manquant : {p.resolve()}\n"
        "Exécutez Spark_DIA3_Session_2.ipynb §2.8 (écriture Parquet) avant ce notebook."
    )

for p in [DELTA_DISPONIBLE, DELTA_ALERTES, STREAM_SOURCE_DIR, STREAM_CHECKPOINT]:
    p.mkdir(parents=True, exist_ok=True)

print(f"[OK] {VELIB_CONSOLIDE.resolve()}")

# ── Paramètres ───────────────────────────────────────────────────────────────
APP_NAME = "ClimaCity-Paris-Jour2"
SHUFFLE_PARTS = 8
SEED = 42


[OK] /Users/romain/Desktop/SparkVelib/data/output/disponibilite_consolidee.parquet


In [36]:
import sys, delta
print(sys.executable)
print(delta.__file__)

/Users/romain/Desktop/SparkVelib/.venv-spark/bin/python
/Users/romain/Desktop/SparkVelib/.venv-spark/lib/python3.12/site-packages/delta/__init__.py


### Reset Spark (en cas d'erreur RPC)

Si vous voyez `RpcEndpointNotFoundException` ou `Issue communicating with driver in heartbeater` :

1. **Kernel → Restart Kernel**
2. Relancez les cellules Section 0 → Reset → SparkSession → `df`


In [37]:
# Reset Spark propre — coupe toute session/driver orphelin dans ce kernel
from pyspark.sql import SparkSession

def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def reset_spark_session() -> None:
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception as exc:
            print(f"spark.stop() ignoré : {exc}")

    if "sc" in globals() and sc is not None:
        try:
            if sc._jsc is not None:
                sc.stop()
        except Exception as exc:
            print(f"sc.stop() ignoré : {exc}")

    _clear_spark_session_registry()
    spark = None
    sc = None
    print("Session Spark réinitialisée. Relancez la cellule SparkSession.")

reset_spark_session()


Session Spark réinitialisée. Relancez la cellule SparkSession.


In [38]:
import os
import sys
import platform
import subprocess
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
from pyspark.sql.window import Window
from delta import configure_spark_with_delta_pip

# Mac Apple Silicon : Java arm64 + workers PySpark arm64
if platform.system() == "Darwin" and platform.machine() == "arm64":
    _jdk_candidates = [
        Path("/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"),
        Path("/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home"),
    ]
    _jdk = next((p for p in _jdk_candidates if (p / "bin" / "java").exists()), None)
    if _jdk:
        os.environ["JAVA_HOME"] = str(_jdk)
        os.environ["PATH"] = f"{_jdk / 'bin'}:{os.environ.get('PATH', '')}"
    _python = Path(sys.executable)
    _wrapper = _python.parent / "python-arm64"
    if not _wrapper.exists():
        _wrapper.write_text(
            f'#!/bin/bash\nexec arch -arm64 "{_python}" "$@"\n',
            encoding="utf-8",
        )
        _wrapper.chmod(0o755)
    os.environ["PYSPARK_PYTHON"] = str(_wrapper)
    os.environ["PYSPARK_DRIVER_PYTHON"] = str(_python)

def _clear_spark_session_registry() -> None:
    from pyspark.sql.context import SQLContext

    SparkSession._instantiatedSession = None
    SparkSession._activeSession = None
    SQLContext._instantiatedContext = None


def _spark_is_alive() -> bool:
    if "spark" not in globals() or spark is None:
        return False
    try:
        _sc = spark.sparkContext
        return _sc is not None and _sc._jsc is not None
    except Exception:
        return False


def reset_spark_session() -> None:
    global spark, sc

    if _spark_is_alive():
        try:
            spark.stop()
        except Exception as exc:
            print(f"spark.stop() ignoré : {exc}")

    if "sc" in globals() and sc is not None:
        try:
            if sc._jsc is not None:
                sc.stop()
        except Exception as exc:
            print(f"sc.stop() ignoré : {exc}")

    _clear_spark_session_registry()
    spark = None
    sc = None
    print("Session Spark réinitialisée. Relancez la cellule SparkSession.")

def _delta_is_configured() -> bool:
    if not _spark_is_alive():
        return False
    try:
        extensions = spark.conf.get("spark.sql.extensions", "")
        catalog = spark.conf.get("spark.sql.catalog.spark_catalog", "")
        return (
            "DeltaSparkSessionExtension" in extensions
            and "DeltaCatalog" in catalog
        )
    except Exception:
        return False


def create_delta_spark_session():
    """Recrée une session Spark avec Delta Lake (obligatoire avant .format("delta"))."""
    global spark, sc

    reset_spark_session()

    _builder = (
        SparkSession.builder
        .appName(APP_NAME)
        .master("local[*]")
        .config("spark.sql.shuffle.partitions", SHUFFLE_PARTS)
        .config("spark.driver.memory", "4g")
        .config("spark.driver.host", "127.0.0.1")
        .config("spark.driver.bindAddress", "127.0.0.1")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
        .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    )
    spark = configure_spark_with_delta_pip(_builder).getOrCreate()
    sc = spark.sparkContext
    sc.setLogLevel("WARN")

    if not _delta_is_configured():
        raise RuntimeError(
            "Delta Lake non actif. Kernel → Restart, puis Section 0 → Reset → cette cellule."
        )

    print(f"Spark {spark.version} -- Delta Lake activé")
    print(f"spark.sql.extensions = {spark.conf.get('spark.sql.extensions')}")
    print(f"Spark UI : http://localhost:4040")
    return spark, sc


# Delta Lake : pip install delta-spark (configure les JARs automatiquement)
# Toujours recréer : getOrCreate() réutiliserait une session Session 1/2 sans Delta.
create_delta_spark_session()


Session Spark réinitialisée. Relancez la cellule SparkSession.
Spark 3.5.9 -- Delta Lake activé
spark.sql.extensions = io.delta.sql.DeltaSparkSessionExtension
Spark UI : http://localhost:4040


26/07/21 14:08:59 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/07/21 14:08:59 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.


(<pyspark.sql.session.SparkSession at 0x125851b20>,
 <SparkContext master=local[*] appName=ClimaCity-Paris-Jour2>)

### Test Spark SQL

Requête minimale pour vérifier que le moteur SQL répond (à exécuter **après** la cellule SparkSession).


In [39]:
spark.sql("SELECT 1 AS ok").show()

+---+
| ok|
+---+
|  1|
+---+



In [40]:
# Chargement de la table consolidée produite au Jour 1
df = spark.read.parquet(str(VELIB_CONSOLIDE))
df.cache()
df.count()   # force la mise en cache

print(f"Table consolidée : {df.count():,} lignes  |  {len(df.columns)} colonnes")
df.printSchema()
#ca nous permettra de connaitre le schema de la table et donc de faire ensuite la requete sql


Table consolidée : 690,858 lignes  |  20 colonnes
root
 |-- station_id: long (nullable = true)
 |-- nom_station: string (nullable = true)
 |-- code_arr: integer (nullable = true)
 |-- capacite: integer (nullable = true)
 |-- horodatage: string (nullable = true)
 |-- velos_meca: integer (nullable = true)
 |-- velos_elec: integer (nullable = true)
 |-- bornettes_libres: integer (nullable = true)
 |-- taux_occupation: double (nullable = true)
 |-- statut: string (nullable = true)
 |-- jour_sem: integer (nullable = true)
 |-- heure: integer (nullable = true)
 |-- est_weekend: boolean (nullable = true)
 |-- temperature_c: double (nullable = true)
 |-- humidite_pct: integer (nullable = true)
 |-- vent_kmh: double (nullable = true)
 |-- precipitation_mm: double (nullable = true)
 |-- est_pluie: boolean (nullable = true)
 |-- annee: integer (nullable = true)
 |-- mois: integer (nullable = true)



---
# PARTIE 1 -- Spark SQL (matin)

## 1.1 Vues temporaires et premières requêtes SQL

L'API DataFrame et Spark SQL sont **entièrement interchangeables** : elles produisent
le même plan d'exécution physique après passage par le Catalyst optimizer.
Le choix entre les deux est une question de lisibilité et d'habitude.

La règle pratique : SQL excelle pour les agrégations complexes et le fenêtrage.
L'API DataFrame est plus commode pour les traitements programmatiques (boucles,
conditions dynamiques, chaînage de transformations).


In [41]:
# Enregistrement des vues temporaires
# Une vue temporaire n'existe que pour la durée de la session Spark.
# Elle ne copie pas les données -- c'est un alias sur le DataFrame.
df.createOrReplaceTempView("disponibilite")

# Vérification
spark.sql("SHOW VIEWS").show()


+---------+-------------+-----------+
|namespace|     viewName|isTemporary|
+---------+-------------+-----------+
|         |disponibilite|       true|
+---------+-------------+-----------+



SHOW VIEWS est une commande Spark SQL (inspirée de Hive/SQL standard) qui liste toutes les vues temporaires enregistrées dans la session Spark courante.

In [42]:
# Première requête : distribution des statuts par arrondissement
# Nombre de snapshots, taux d'occupation moyen, écart-type de l'occupation
spark.sql("""
SELECT
    code_arr,
    statut,
    COUNT(*)                    AS nb_snapshots,
    ROUND(AVG(taux_occupation), 4) AS taux_moyen,
    ROUND(STDDEV(taux_occupation), 4) AS ecart_type
FROM disponibilite
WHERE code_arr IS NOT NULL
GROUP BY code_arr, statut
ORDER BY code_arr, statut
""").show(40)


+--------+------+------------+----------+----------+
|code_arr|statut|nb_snapshots|taux_moyen|ecart_type|
+--------+------+------------+----------+----------+
|       6|  vide|        4666|    0.0621|    0.0198|
|       7|  vide|        3141|     0.066|    0.0165|
|      29|  vide|          81|    0.0829|     0.021|
|      30|  vide|         943|    0.0658|    0.0202|
|      36|  vide|        1688|    0.0675|    0.0196|
|      37|  vide|         521|     0.057|    0.0149|
|      38|  vide|        3770|    0.0654|    0.0223|
|      52|  vide|        6417|      0.06|    0.0205|
|      63|  vide|        1186|     0.069|    0.0166|
|      77|  vide|         229|    0.0622|    0.0156|
|   10968|  vide|         454|    0.0526|       0.0|
|   11375|  vide|         702|    0.0621|    0.0199|
|   11580|  vide|          71|    0.0588|       0.0|
|   13105|  vide|        1189|    0.0711|    0.0238|
|   13191|  vide|        1176|    0.0638|    0.0257|
|   13242|  vide|        2464|    0.0652|    0

In [43]:
# Les fonctions temporelles SQL sont disponibles directement
# Taux moyen d'occupation, nombre de snapshots, nombre de stations par heure
spark.sql("""
SELECT
    HOUR(horodatage)                       AS heure,
    COUNT(*)                               AS nb_snapshots,
    COUNT(DISTINCT nom_station)            AS nb_stations,
    ROUND(AVG(taux_occupation), 4)         AS taux_moyen
FROM disponibilite
GROUP BY HOUR(horodatage)
ORDER BY heure
""").show(24)


+-----+------------+-----------+----------+
|heure|nb_snapshots|nb_stations|taux_moyen|
+-----+------------+-----------+----------+
| NULL|      690858|       1317|    0.0604|
+-----+------------+-----------+----------+



---
## 1.2 Questions métier -- Requêtes analytiques

L'équipe métier a soumis trois questions auxquelles votre plateforme doit répondre.
Nous allons les traiter une par une avec Spark SQL.


### Question 1 : Ruptures en heure de pointe matinale

> Quelles sont les 10 stations les plus souvent en rupture totale (zéro vélo disponible,
> mécanique ou électrique) entre 7 h et 10 h, les jours de semaine,
> en excluant les jours fériés français ?

Les jours fériés français sont injectés comme une petite table de référence --
c'est l'occasion d'illustrer la `broadcast join` en SQL.


In [44]:
# Table de jours fériés (2022-2023) -- injectée en broadcast
# Source : légifrance.gouv.fr
jours_feries = spark.createDataFrame([
    ("2022-01-01",), ("2022-04-18",), ("2022-05-01",), ("2022-05-08",),
    ("2022-05-26",), ("2022-06-06",), ("2022-07-14",), ("2022-08-15",),
    ("2022-11-01",), ("2022-11-11",), ("2022-12-25",),
    ("2023-01-01",), ("2023-04-10",), ("2023-05-01",), ("2023-05-08",),
    ("2023-05-18",), ("2023-05-29",), ("2023-07-14",), ("2023-08-15",),
    ("2023-11-01",), ("2023-11-11",), ("2023-12-25",),
], ["date_ferie"])

jours_feries = jours_feries.withColumn(
    "date_ferie", F.to_date("date_ferie", "yyyy-MM-dd")
)
jours_feries.createOrReplaceTempView("jours_feries")

print(f"{jours_feries.count()} jours fériés enregistrés (2022-2023)")


22 jours fériés enregistrés (2022-2023)


In [45]:
# Identifier les 10 stations Vélib' les plus en rupture de stock pendant les heures de pointe matinales, en excluant les jours fériés.
# On ne s'intéresse qu'aux stations très fréquentées, ayant plus de 100 observations (snapshots)
# -> Id de la station, nom de la station, arrondissement, nombre d'observations (snapshots)
df_q1 = spark.sql("""
WITH disponibilite_ts AS (
    SELECT
        station_id,
        nom_station,
        code_arr,
        statut,
        TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm'Z'") AS ts
    FROM disponibilite
)
SELECT
    d.station_id,
    d.nom_station,
    d.code_arr,
    COUNT(*) AS nb_snapshots
FROM disponibilite_ts d
LEFT ANTI JOIN jours_feries jf
    ON TO_DATE(d.ts) = jf.date_ferie
WHERE d.ts IS NOT NULL
  AND HOUR(d.ts) BETWEEN 7 AND 9
  AND d.statut = 'vide'
GROUP BY d.station_id, d.nom_station, d.code_arr
HAVING COUNT(*) > 100
ORDER BY nb_snapshots DESC, d.station_id
LIMIT 10
""")

df_q1.show(truncate=False)

+----------+----------------------------------------+--------+------------+
|station_id|nom_station                             |code_arr|nb_snapshots|
+----------+----------------------------------------+--------+------------+
|NULL      |Chateaubriand - Friedland               |NULL    |366         |
|213992316 |Square du Théâtre du Garde-Chasse       |213992  |321         |
|213682736 |Morillons - Dantzig                     |213682  |316         |
|653076373 |Bois de Vincennes - Porte Jaune         |653076  |311         |
|256124192 |Belleville - Pyrénées                   |256124  |291         |
|213688824 |Faustin Hélie - Desbordes-Valmore       |213688  |289         |
|129086648 |Ceinture du Lac Inferieure - Saint-Cloud|129086  |288         |
|124402232 |Belleville - Télégraphe                 |124402  |279         |
|653104215 |Sorbier - Gasnier-Guy                   |653104  |279         |
|15462861  |Crevaux - Bugeaud                       |15462   |276         |
+----------+

### Question 2 : Impact de la pluie sur le taux d'occupation

> La pluie réduit-elle statistiquement le taux d'occupation moyen du réseau ?
> De combien de points en moyenne ? L'effet est-il homogène selon les arrondissements ?


In [49]:
# Distribution statistique par quartiles du taux d'occupation en fonction de la météo
# LEFT JOIN explicite : compare sec / pluie (est_pluie NOT NULL) et jointure absente (est_pluie IS NULL)
METEO_CSV = DATA_DIR / "meteo" / "paris_montsouris_horaire.csv"
(
    spark.read
    .option("sep", ";")
    .option("header", True)
    .option("inferSchema", True)
    .csv(str(METEO_CSV))
    .createOrReplaceTempView("meteo")
)

df_q2 = spark.sql("""
WITH disponibilite_h AS (
    SELECT
        taux_occupation,
        DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm'Z'")) AS heure_tronquee
    FROM disponibilite
),
meteo_h AS (
    SELECT
        DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm")) AS heure_tronquee,
        temperature                                              AS temperature_c,
        humidite                                                 AS humidite_pct,
        vent_ms * 3.6                                            AS vent_kmh,
        precipitations                                           AS precipitation_mm,
        CASE WHEN precipitations > 0 THEN true ELSE false END    AS est_pluie
    FROM meteo
),
meteo_occupation AS (
    SELECT
        v.taux_occupation,
        m.est_pluie,
        m.temperature_c,
        m.humidite_pct,
        m.vent_kmh,
        m.precipitation_mm
    FROM disponibilite_h v
    LEFT JOIN meteo_h m
        ON v.heure_tronquee = m.heure_tronquee
)
SELECT
    CASE
        WHEN est_pluie IS NULL THEN 'jointure_meteo_absente'
        WHEN est_pluie THEN 'pluie'
        ELSE 'sec'
    END                                                   AS condition_meteo,
    est_pluie,
    COUNT(*)                                              AS nb_snapshots,
    ROUND(AVG(taux_occupation), 4)                        AS taux_moyen,
    ROUND(STDDEV(taux_occupation), 4)                     AS ecart_type,
    ROUND(percentile_approx(taux_occupation, 0.25), 4)    AS Q1,
    ROUND(percentile_approx(taux_occupation, 0.50), 4)    AS mediane,
    ROUND(percentile_approx(taux_occupation, 0.75), 4)    AS Q3,
    ROUND(AVG(temperature_c), 2)                          AS temp_moy_c,
    ROUND(AVG(humidite_pct), 1)                           AS humidite_moy_pct,
    ROUND(AVG(vent_kmh), 2)                               AS vent_moy_kmh,
    ROUND(AVG(precipitation_mm), 2)                       AS precip_moy_mm,
    ROUND(MAX(precipitation_mm), 2)                       AS precipitation_max_mm
FROM meteo_occupation
GROUP BY
    CASE
        WHEN est_pluie IS NULL THEN 'jointure_meteo_absente'
        WHEN est_pluie THEN 'pluie'
        ELSE 'sec'
    END,
    est_pluie
ORDER BY condition_meteo
""")
df_q2.show(truncate=False)


+---------------+---------+------------+----------+----------+------+-------+----+----------+----------------+------------+-------------+--------------------+
|condition_meteo|est_pluie|nb_snapshots|taux_moyen|ecart_type|Q1    |mediane|Q3  |temp_moy_c|humidite_moy_pct|vent_moy_kmh|precip_moy_mm|precipitation_max_mm|
+---------------+---------+------------+----------+----------+------+-------+----+----------+----------------+------------+-------------+--------------------+
|pluie          |true     |168618      |0.0609    |0.022     |0.0417|0.0606 |0.08|7.86      |89.9            |18.31       |0.48         |3.7                 |
|sec            |false    |522240      |0.0602    |0.022     |0.0417|0.06   |0.08|5.02      |88.0            |13.19       |0.0          |0.0                 |
+---------------+---------+------------+----------+----------+------+-------+----+----------+----------------+------------+-------------+--------------------+



In [51]:
# Effet de la pluie par arrondissement
# Prérequis : cellule df_q2 exécutée (vue temporaire meteo)
df_q2_arr = spark.sql("""
WITH disponibilite_h AS (
    SELECT
        code_arr,
        taux_occupation,
        DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm'Z'")) AS heure_tronquee
    FROM disponibilite
    WHERE code_arr IS NOT NULL
),
meteo_h AS (
    SELECT
        DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm")) AS heure_tronquee,
        CASE WHEN precipitations > 0 THEN true ELSE false END AS est_pluie
    FROM meteo
),
meteo_par_arr AS (
    SELECT
        d.code_arr,
        d.taux_occupation,
        m.est_pluie
    FROM disponibilite_h d
    LEFT JOIN meteo_h m
        ON d.heure_tronquee = m.heure_tronquee
    WHERE m.est_pluie IS NOT NULL
)
SELECT
    code_arr,
    COUNT(*) AS nb_snapshots,
    ROUND(AVG(CASE WHEN est_pluie = false THEN taux_occupation END), 4) AS taux_moyen_sec,
    ROUND(AVG(CASE WHEN est_pluie = true  THEN taux_occupation END), 4) AS taux_moyen_pluie,
    ROUND(
        AVG(CASE WHEN est_pluie = true  THEN taux_occupation END)
        - AVG(CASE WHEN est_pluie = false THEN taux_occupation END),
        4
    ) AS delta_pluie_vs_sec
FROM meteo_par_arr
GROUP BY code_arr
HAVING COUNT(CASE WHEN est_pluie = false THEN 1 END) > 50
   AND COUNT(CASE WHEN est_pluie = true  THEN 1 END) > 50
ORDER BY delta_pluie_vs_sec, code_arr
""")
df_q2_arr.show(30)
# Un delta négatif signifie que le taux d'occupation baisse sous la pluie
# (moins de vélos empruntés -> plus de vélos disponibles -> moins de bornettes libres)


+--------+------------+--------------+----------------+------------------+
|code_arr|nb_snapshots|taux_moyen_sec|taux_moyen_pluie|delta_pluie_vs_sec|
+--------+------------+--------------+----------------+------------------+
|  129026|         214|        0.0766|          0.0623|           -0.0142|
|  523900|         989|        0.0431|          0.0297|           -0.0134|
| 1074398|         277|          0.07|          0.0581|            -0.012|
|  101759|         260|        0.0665|          0.0556|           -0.0108|
|  653114|         334|        0.0658|          0.0567|           -0.0091|
|   53999|         325|        0.0732|          0.0645|           -0.0087|
|  205047|         996|        0.0618|          0.0534|           -0.0085|
|   66489|         383|        0.0689|          0.0606|           -0.0083|
|  121086|         632|        0.0758|          0.0675|           -0.0083|
| 1056965|        1788|        0.0555|          0.0472|           -0.0083|
|  210582|         291|  

### Question 3 : Saisonnalité intra-journalière

> Quelle station présente la plus forte amplitude entre son heure creuse et son heure
> de pointe au cours d'une journée type (taux_max - taux_min par heure) ?

C'est un cas d'usage typique des **fonctions de fenêtrage**.


In [52]:
# Quelles sont les 15 stations dont le comportement est le plus pendulaire — presque vides à certaines heures, saturées à d'autres ?
df_q3 = spark.sql("""
WITH taux_horaire AS (
    SELECT
        station_id,
        nom_station,
        heure,
        AVG(taux_occupation) AS taux_moyen_h
    FROM disponibilite
    WHERE heure IS NOT NULL
      AND station_id IS NOT NULL
    GROUP BY station_id, nom_station, heure
),
amplitude_station AS (
    SELECT
        station_id,
        nom_station,
        MAX(taux_moyen_h) AS taux_max_h,
        MIN(taux_moyen_h) AS taux_min_h,
        MAX(taux_moyen_h) - MIN(taux_moyen_h) AS amplitude
    FROM taux_horaire
    GROUP BY station_id, nom_station
)
SELECT
    station_id,
    nom_station,
    ROUND(taux_min_h, 4) AS taux_min_horaire,
    ROUND(taux_max_h, 4) AS taux_max_horaire,
    ROUND(amplitude, 4) AS amplitude
FROM amplitude_station
ORDER BY amplitude DESC, station_id
LIMIT 15
""")
df_q3.show(truncate=False)


+----------+-------------------------------------+----------------+----------------+---------+
|station_id|nom_station                          |taux_min_horaire|taux_max_horaire|amplitude|
+----------+-------------------------------------+----------------+----------------+---------+
|39254792  |Mairie du 11ème                      |0.0196          |0.098           |0.0784   |
|266963426 |Beaumarchais - Saint-Gilles          |0.0196          |0.098           |0.0784   |
|94250567  |Mairie d'Ivry-sur-Seine              |0.0189          |0.0943          |0.0754   |
|54000610  |Jean de la Fontaine - Boulainvilliers|0.0147          |0.0882          |0.0735   |
|1056986761|Terroirs de France - Lheureux        |0.0182          |0.0909          |0.0727   |
|49341735  |Centquatre                           |0.0244          |0.097           |0.0726   |
|42499285  |Petit Palais                         |0.0233          |0.093           |0.0697   |
|437741660 |Saint-Denis - Rivoli                 |

In [54]:
# Inspecter le format brut des horodatages
spark.sql("""
    SELECT horodatage
    FROM disponibilite
    LIMIT 5
""").show(truncate=False)

+-----------------+
|horodatage       |
+-----------------+
|2020-12-04T03:07Z|
|2020-12-04T03:07Z|
|2020-12-04T03:07Z|
|2020-12-04T03:07Z|
|2020-12-04T03:07Z|
+-----------------+



In [55]:
spark.sql("""
    SELECT horodatage
    FROM meteo
    LIMIT 5
""").show(truncate=False)

+-------------------+
|horodatage         |
+-------------------+
|2020-01-01 00:00:00|
|2020-01-01 01:00:00|
|2020-01-01 02:00:00|
|2020-01-01 03:00:00|
|2020-01-01 04:00:00|
+-------------------+



In [53]:
# [EXERCICE]
# En utilisant Spark SQL, calculez pour chaque station :
# - le taux d'occupation moyen un jour de semaine sec (est_pluie = false)
# - le taux d'occupation moyen un week-end pluvieux (est_pluie = true)
# - le ratio entre les deux
# Affichez les 10 stations avec le ratio le plus élevé (plus forte différence).
#
# Rappel : est_weekend est un booléen, est_pluie aussi.
# ──────────────────────────────────────────────────────────────────────────

# Prérequis : cellule df_q2 exécutée (vue temporaire meteo)
spark.sql("""
WITH disponibilite_meteo AS (
    SELECT
        d.station_id,
        d.nom_station,
        d.taux_occupation,
        d.est_weekend,
        m.est_pluie
    FROM (
        SELECT
            station_id,
            nom_station,
            taux_occupation,
            est_weekend,
            DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm'Z'")) AS heure_tronquee
        FROM disponibilite
        WHERE station_id IS NOT NULL
    ) d
    LEFT JOIN (
        SELECT
            DATE_TRUNC('hour', TO_TIMESTAMP(horodatage, "yyyy-MM-dd'T'HH:mm")) AS heure_tronquee,
            CASE WHEN precipitations > 0 THEN true ELSE false END AS est_pluie
        FROM meteo
    ) m ON d.heure_tronquee = m.heure_tronquee
    WHERE m.est_pluie IS NOT NULL
),
stats_station AS (
    SELECT
        station_id,
        nom_station,
        AVG(CASE WHEN est_weekend = false AND est_pluie = false THEN taux_occupation END) AS taux_semaine_sec,
        AVG(CASE WHEN est_weekend = true  AND est_pluie = true  THEN taux_occupation END) AS taux_weekend_pluie,
        COUNT(CASE WHEN est_weekend = false AND est_pluie = false THEN 1 END) AS n_semaine_sec,
        COUNT(CASE WHEN est_weekend = true  AND est_pluie = true  THEN 1 END) AS n_weekend_pluie
    FROM disponibilite_meteo
    GROUP BY station_id, nom_station
)
SELECT
    station_id,
    nom_station,
    ROUND(taux_semaine_sec, 4) AS taux_semaine_sec,
    ROUND(taux_weekend_pluie, 4) AS taux_weekend_pluie,
    ROUND(taux_weekend_pluie / taux_semaine_sec, 4) AS ratio
FROM stats_station
WHERE taux_semaine_sec IS NOT NULL
  AND taux_weekend_pluie IS NOT NULL
  AND taux_semaine_sec > 0
  AND n_semaine_sec >= 10
  AND n_weekend_pluie >= 10
ORDER BY ratio DESC
LIMIT 10
""").show(truncate=False)


+----------+---------------------------------+----------------+------------------+------+
|station_id|nom_station                      |taux_semaine_sec|taux_weekend_pluie|ratio |
+----------+---------------------------------+----------------+------------------+------+
|66505513  |François 1er - Lincoln           |0.0645          |0.0968            |1.5   |
|205938683 |Delphine Seyrig - Général Leclerc|0.0535          |0.0779            |1.4549|
|66491368  |Sorbonne - Ecoles                |0.0552          |0.08              |1.4497|
|128843085 |Place du Lieutenant Henri Karcher|0.0464          |0.0667            |1.4382|
|101015227 |Chevaleret - Louise Weiss        |0.0511          |0.0727            |1.4218|
|76352106  |Saint-Augustin                   |0.0666          |0.0936            |1.4041|
|653197206 |Marcel Pourtout - Édouard Manet  |0.0596          |0.0833            |1.3982|
|653106960 |Gare RER - Maurice Grandcoing    |0.0574          |0.0789            |1.3763|
|129164969

---
## 1.3 Fonctions de fenêtrage analytique

Les fonctions de fenêtrage (`WINDOW` / `OVER`) permettent de calculer des agrégats
**sans réduire le nombre de lignes** -- contrairement à `GROUP BY`.
Elles sont indispensables pour les analyses de séries temporelles.

### Les trois familles de fonctions fenêtrées

```
Ranking    : ROW_NUMBER, RANK, DENSE_RANK, NTILE
Navigation : LAG, LEAD, FIRST_VALUE, LAST_VALUE, NTH_VALUE
Agrégation : SUM, AVG, MIN, MAX, COUNT (avec clause OVER)
```


In [56]:
# Cas concret : pour chaque station, calculer le taux d'occupation
# de la fenêtre précédente (LAG) et suivante (LEAD),
# ainsi qu'une moyenne mobile sur 3 snapshots.

fenetre_station = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
)

fenetre_moy_mobile_3 = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
    .rowsBetween(-2, Window.currentRow)  # snapshot courant + 2 précédents = 3 lignes
)

df_avec_lag = (
    df
    .select("station_id", "nom_station", "horodatage", "taux_occupation")
    .withColumn("taux_precedent", F.lag("taux_occupation", 1).over(fenetre_station))
    .withColumn("taux_suivant", F.lead("taux_occupation", 1).over(fenetre_station))
    .withColumn("moy_mobile_3", F.round(F.avg("taux_occupation").over(fenetre_moy_mobile_3), 4))
)
df_avec_lag.show(truncate=False)


+----------+--------------------------+-----------------+---------------+--------------+------------+------------+
|station_id|nom_station               |horodatage       |taux_occupation|taux_precedent|taux_suivant|moy_mobile_3|
+----------+--------------------------+-----------------+---------------+--------------+------------+------------+
|7402      |Roquette - Auguste Laurent|2020-12-04T06:33Z|0.0625         |NULL          |0.0625      |0.0625      |
|7402      |Roquette - Auguste Laurent|2020-12-04T06:52Z|0.0625         |0.0625        |0.0625      |0.0625      |
|7402      |Roquette - Auguste Laurent|2020-12-05T14:47Z|0.0625         |0.0625        |0.0625      |0.0625      |
|7402      |Roquette - Auguste Laurent|2020-12-05T15:29Z|0.0625         |0.0625        |0.0625      |0.0625      |
|7402      |Roquette - Auguste Laurent|2020-12-06T12:54Z|0.0625         |0.0625        |0.0625      |0.0625      |
|7402      |Roquette - Auguste Laurent|2020-12-06T14:28Z|0.0625         |0.0625 

In [57]:
# Classement des stations par taux d'occupation moyen, à chaque heure de la journée
# ROW_NUMBER() numérote les lignes dans chaque partition (ici : chaque heure)

fenetre_heure = (
    Window
    .partitionBy("heure")
    .orderBy(F.col("taux_moyen").desc())
)

df_rank = (
    df
    .groupBy("station_id", "nom_station", "heure")
    .agg(F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"))
    .withColumn("rang", F.row_number().over(fenetre_heure))
    .orderBy("heure", "rang")
)
df_rank.show(48, truncate=False)


+----------+------------------------------------+-----+----------+----+
|station_id|nom_station                         |heure|taux_moyen|rang|
+----------+------------------------------------+-----+----------+----+
|213868408 |Gare Routière de Saint Cloud        |0    |0.0968    |1   |
|37952191  |Jean Jaurès - Reine                 |0    |0.0968    |2   |
|1036199130|Les Juilliottes                     |0    |0.0952    |3   |
|27414544  |Place Monge                         |0    |0.0952    |4   |
|15309674  |Square Montholon                    |0    |0.0952    |5   |
|27269170  |Quai aux Fleurs - Pont Saint-Louis  |0    |0.0952    |6   |
|1055541632|Place Tony de Graaff                |0    |0.0952    |7   |
|210386007 |Alger - Rivoli                      |0    |0.0952    |8   |
|129130397 |Pierre Grenier - Pont de Billancourt|0    |0.0952    |9   |
|66491404  |Flandre - Riquet                    |0    |0.0952    |10  |
|29327     |Stade Auguste Delaune               |0    |0.0952   

In [58]:
# Calcul de la variation du taux sur une heure glissante
# UNBOUNDED PRECEDING -> ligne actuelle = cumul depuis le début de la partition

fenetre_cumul = (
    Window
    .partitionBy("station_id")
    .orderBy("horodatage")
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)
)

# Variation par rapport au snapshot précédent (delta instantané)
station_cible = 1042   # à adapter selon vos données

df_delta = (
    df # requête sous forme fonctionnelle
)
df_delta.show(truncate=False)


+----------+------------------------------------+--------+--------+-----------------+----------+----------+----------------+---------------+------+--------+-----+-----------+-------------+------------+--------+----------------+---------+-----+----+
|station_id|nom_station                         |code_arr|capacite|horodatage       |velos_meca|velos_elec|bornettes_libres|taux_occupation|statut|jour_sem|heure|est_weekend|temperature_c|humidite_pct|vent_kmh|precipitation_mm|est_pluie|annee|mois|
+----------+------------------------------------+--------+--------+-----------------+----------+----------+----------------+---------------+------+--------+-----+-----------+-------------+------------+--------+----------------+---------+-----+----+
|NULL      |Cimetière                           |NULL    |15      |2020-12-04T03:07Z|0         |1         |14              |0.0667         |vide  |6       |4    |false      |NULL         |NULL        |NULL    |NULL            |NULL     |2020 |12  |
|596

---
## 1.4 Delta Lake : transactions, time-travel et MERGE

Delta Lake est une couche de stockage transactionnel construite par-dessus Parquet.
Elle apporte à Spark les propriétés **ACID** qui manquent au Parquet brut :

| Propriété | Parquet brut | Delta Lake |
|-----------|-------------|------------|
| Lecture cohérente pendant une écriture | Non | Oui (MVCC) |
| Annulation d'une écriture partielle | Non | Oui |
| Historique des versions | Non | Oui (time-travel) |
| Mise à jour / suppression de lignes | Non | Oui (MERGE, UPDATE, DELETE) |
| Optimisation automatique | Non | Oui (OPTIMIZE, Z-ORDER) |

### Écriture en format Delta


In [59]:
from delta.tables import DeltaTable

if not _delta_is_configured():
    print("Session Spark sans Delta — recréation…")
    create_delta_spark_session()
    df = spark.read.parquet(str(VELIB_CONSOLIDE))
    df.cache()
    df.createOrReplaceTempView("disponibilite")

# Écriture initiale : données 2020 (Velib historique du projet)
df_2020 = df.filter(F.col("annee") == 2020)

t0 = time.perf_counter()
(
    df_2020
    .write
    .format("delta")
    .mode("overwrite")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Écriture 2020 : {time.perf_counter()-t0:.1f} s  --  {df_2020.count():,} lignes")

# Vérification de la structure Delta
import os
fichiers_delta = list(Path(DELTA_DISPONIBLE).rglob("*.parquet"))
log_delta      = list(Path(DELTA_DISPONIBLE / "_delta_log").glob("*.json"))
print(f"Fichiers Parquet : {len(fichiers_delta)}")
print(f"Entrées dans le transaction log : {len(log_delta)}")


Écriture 2020 : 1.5 s  --  460,591 lignes
Fichiers Parquet : 6
Entrées dans le transaction log : 1


In [60]:
# Ajout des données 2021 -- mode "append"
df_2021 = df.filter(F.col("annee") == 2021)

t0 = time.perf_counter()
(
    df_2021
    .write
    .format("delta")
    .mode("append")
    .partitionBy("annee", "mois")
    .save(str(DELTA_DISPONIBLE))
)
print(f"Ajout 2021 : {time.perf_counter()-t0:.1f} s  --  {df_2021.count():,} lignes")

# Historique des versions : chaque opération crée une nouvelle version
delta_table = DeltaTable.forPath(spark, str(DELTA_DISPONIBLE))
delta_table.history().select(
    "version", "timestamp", "operation", "operationParameters"
).show(truncate=False)


Ajout 2021 : 0.0 s  --  230,267 lignes
+-------+-----------------------+---------+----------------------------------------------------+
|version|timestamp              |operation|operationParameters                                 |
+-------+-----------------------+---------+----------------------------------------------------+
|0      |2026-07-21 14:54:48.856|WRITE    |{mode -> Overwrite, partitionBy -> ["annee","mois"]}|
+-------+-----------------------+---------+----------------------------------------------------+



### Time-travel : interroger une version passée

Delta Lake conserve toutes les versions de la table dans le transaction log.
On peut interroger n'importe quelle version passée avec la clause
`VERSION AS OF` ou `TIMESTAMP AS OF`.


In [62]:
# Lecture de la version 0 (uniquement les données 2020)
df_v0 = (
    spark.read
    .format("delta")
    .option("versionAsOf", 0)
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version 0 (2020 uniquement) : {df_v0.count():,} lignes")

# Lecture de la version courante
df_current = (
    spark.read
    .format("delta")
    .load(str(DELTA_DISPONIBLE))
)
print(f"Version courante (2020+2021) : {df_current.count():,} lignes")

# Enregistrement comme vue SQL pour la suite
df_current.createOrReplaceTempView("disponibilite_delta")


26/07/21 14:56:06 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


Version 0 (2020 uniquement) : 460,591 lignes
Version courante (2020+2021) : 460,591 lignes


### `MERGE INTO` : mise à jour incrémentale

`MERGE INTO` est l'opération la plus puissante de Delta Lake. Elle permet de
**synchroniser** une table cible avec une table source en une seule passe :
insertions des nouvelles lignes, mises à jour des lignes existantes,
suppressions optionnelles.

Cas d'usage typique : arrivée quotidienne d'un nouveau batch de snapshots.


In [65]:
# Simulation : un nouveau batch arrive avec des corrections
# (quelques lignes modifiées + quelques nouvelles lignes)
from pyspark.sql.functions import lit

# On prend 500 snapshots existants et on simule une correction du taux_occupation
df_corrections = (
    df_current
    .filter(F.col("mois") == 1)
    .limit(500)
    .withColumn("taux_occupation", F.round(F.col("taux_occupation") * 0.98, 4))
    .withColumn("source", lit("correction_batch"))
)

# Quelques nouvelles lignes fictives (snapshots manqués, horodatages décalés de 2 ans)
df_nouveaux = (
    df_current
    .filter(F.col("mois") == 1)
    .orderBy("station_id", "horodatage")
    .offset(500)
    .limit(50)
    .withColumn(
        "horodatage",
        F.expr(
            "date_format("
            "to_timestamp(horodatage, \"yyyy-MM-dd'T'HH:mm'Z'\") + INTERVAL 2 YEARS, "
            "\"yyyy-MM-dd'T'HH:mm'Z'\""
            ")"
        ),
    )
    .withColumn("annee", lit(2022))
    .withColumn("mois", lit(1))
    .withColumn("source", lit("nouveau_batch"))
)

df_batch = df_corrections.unionByName(df_nouveaux).drop("source")
print(
    f"Batch entrant : {df_batch.count()} lignes "
    f"({df_corrections.count()} corrections + {df_nouveaux.count()} nouvelles)"
)


Batch entrant : 0 lignes (0 corrections + 0 nouvelles)


In [66]:
# MERGE INTO : upsert (update + insert)
(
    delta_table.alias("cible")
    .merge(
        df_batch.alias("source"),
        # Condition de correspondance : même station, même horodatage
        "cible.station_id = source.station_id AND cible.horodatage = source.horodatage"
    )
    .whenMatchedUpdateAll()     # si correspondance : on écrase toutes les colonnes
    .whenNotMatchedInsertAll()  # si pas de correspondance : on insère
    .execute()
)

# Vérification : la table a une nouvelle version
delta_table.history().select(
    "version", "timestamp", "operation",
    "operationMetrics"
).show(5, truncate=False)


+-------+-----------------------+---------+-------------------------------------------------------------------+
|version|timestamp              |operation|operationMetrics                                                   |
+-------+-----------------------+---------+-------------------------------------------------------------------+
|0      |2026-07-21 14:54:48.856|WRITE    |{numFiles -> 6, numOutputRows -> 460591, numOutputBytes -> 3336569}|
+-------+-----------------------+---------+-------------------------------------------------------------------+



In [70]:
# [EXERCICE]
# Le MERGE précédent a mis à jour des lignes existantes.
# Utilisez le time-travel pour comparer le taux_occupation moyen
# de janvier 2022 AVANT et APRÈS le merge.
#
# Indice : lisez la version juste avant le MERGE et la version courante,
# puis comparez avec une agrégation.
# Prérequis : cellules « append 2021 » puis « MERGE INTO » exécutées.
# ──────────────────────────────────────────────────────────────────────────

filtre_jan_2022 = (F.col("annee") == 2022) & (F.col("mois") == 1)

# Historique Delta : version immédiatement avant le dernier MERGE
# (souvent v1 si overwrite v0 + append v1 + merge v2)
derniere_op = (
    delta_table.history()
    .orderBy(F.col("version").desc())
    .select("version", "operation")
    .first()
)

if derniere_op.operation == "MERGE":
    version_avant_merge = derniere_op.version - 1
else:
    version_avant_merge = derniere_op.version
    print(
        f"Avertissement : dernière opération = '{derniere_op.operation}' "
        f"(version {derniere_op.version}). Exécutez d'abord append 2021 puis MERGE."
    )

print(f"Time-travel : version {version_avant_merge} (avant) vs courante (après)")

df_avant_merge = (
    spark.read
    .format("delta")
    .option("versionAsOf", version_avant_merge)
    .load(str(DELTA_DISPONIBLE))
)

df_apres_merge = (
    spark.read
    .format("delta")
    .load(str(DELTA_DISPONIBLE))
)

(
    df_avant_merge
    .filter(filtre_jan_2022)
    .agg(
        F.lit(f"avant_merge (version {version_avant_merge})").alias("moment"),
        F.count("*").alias("nb_lignes"),
        F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"),
    )
    .unionByName(
        df_apres_merge
        .filter(filtre_jan_2022)
        .agg(
            F.lit("apres_merge (version courante)").alias("moment"),
            F.count("*").alias("nb_lignes"),
            F.round(F.avg("taux_occupation"), 4).alias("taux_moyen"),
        )
    )
    .orderBy("moment")
    .show(truncate=False)
)


Avertissement : dernière opération = 'WRITE' (version 0). Exécutez d'abord append 2021 puis MERGE.
Time-travel : version 0 (avant) vs courante (après)
+------------------------------+---------+----------+
|moment                        |nb_lignes|taux_moyen|
+------------------------------+---------+----------+
|apres_merge (version courante)|0        |NULL      |
|avant_merge (version 0)       |0        |NULL      |
+------------------------------+---------+----------+

